In [1]:
from utils import *

#establish planet calls

if os.environ.get('PL_API_KEY', ''):
    API_KEY = os.environ.get('PL_API_KEY', '')
else:
    API_KEY = '1c61ae19230448e19b5bc359cbf1232e'
    
session = requests.Session()
session.auth = (API_KEY, "")
from IPython import get_ipython

sites_total = [
    ('arrowhead_beach', 36.235731, -76.701847),
    ('cannons_ferry', 36.272006, -76.675842),
    ('rocky_hock_lane', 36.184133, -76.722858),
    ('chowan_river', 36.055163, -76.69076),
    ('point_comfort', 36.169908, -76.745905),
    ('mt_gould', 36.125495, -76.740141),
    ('mid_chowan_river', 36.20983, -76.72677),
    ('north_chowan_river', 36.3236, -76.73354),
    ('edenton_bay_dock', 36.055382, -76.610319),
    ('edenhouse', 36.0476, -76.69611),
    ('albemarle_sound', 35.99002, -76.6092)
]

/Users/sarah/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
def download_satellite(json_dir, download_path, x, y, cc, input_date,sites_total, API_KEY):
    item_type = "PSScene"
    image_ids_set = set()

    for file in os.listdir(json_dir):
        with open(os.path.join(json_dir, file), 'r') as f:
            data = json.load(f)

            date_obj = datetime.strptime(input_date, "%Y-%m-%d")
            date_2_days_prior = date_obj - timedelta(days=2)
            date_lte = date_obj.strftime('%Y-%m-%dT') + '00:00:00.000Z'
            date_gte = date_2_days_prior.strftime('%Y-%m-%dT') + '00:00:00.000Z'

            geojson_geometry = {
                "type": "Polygon",
                "coordinates": data['features'][0]['geometry']['coordinates']}

            combined_filter = {
                "type": "AndFilter",
                "config": [
                    {"type": "GeometryFilter", "field_name": "geometry", "config": geojson_geometry},
                    {"type": "DateRangeFilter", "field_name": "acquired", "config": {"gte": date_gte, "lte": date_lte}},
                    {"type": "RangeFilter", "field_name": "cloud_cover", "config": {"lte": cc}},
                    {"type": "StringInFilter", "field_name": "quality_category", "config": ["standard"]}
                ]
            }

            search_request = {
                "item_types": ["PSScene"],
                "filter": combined_filter
            }

            response = requests.post(
                'https://api.planet.com/data/v1/quick-search',
                auth=HTTPBasicAuth(API_KEY, ''),
                json=search_request
            )

            results = response.json()

            current_ids = [feature['id'] for feature in results.get('features', [])]

#             print("Found in this file:", current_ids)

            for img_id in current_ids:
                if img_id not in image_ids_set:
                    print(f"Adding {img_id}")
                image_ids_set.add(img_id)

    print("Total unique image_ids:", len(image_ids_set))

    imagelinks_delayed(image_ids_set,item_type,API_KEY)
    
    download_links = {}
    for image_id in list(image_ids_set)[:]:
        id_url = 'https://api.planet.com/data/v1/item-types/{}/items/{}/assets'.format(item_type, image_id)

        # Returns JSON metadata for assets in this ID.
        result = requests.get(id_url, auth=HTTPBasicAuth(API_KEY, ''))
        asset_info = result.json().get("ortho_visual")  # Replace with the appropriate asset type

        if asset_info:
            links = asset_info["_links"]
            self_link = links["_self"]
            activation_link = links["activate"]

            # Request activation of the asset:
            activate_result = requests.get(activation_link, auth=HTTPBasicAuth(API_KEY, ''))
            activation_status_result = requests.get(self_link, auth=HTTPBasicAuth(API_KEY, ''))

            activation_status = activation_status_result.json()
            download_link = activation_status.get("location")

            if download_link:

                download_links[image_id] = download_link
                print(f"Download link for image ID {image_id}: {download_link}")
            else:
                print(f"Activation status for image ID {image_id} is not available.")

        else:
            print(f"Asset 'ortho_visual' not found for image ID {image_id}")
    image_download(download_path, download_links)
    crop_and_download(x, y, download_path, sites_total)

In [3]:
# import schedule
# import time

# def func():
#     print("Geeksforgeeks")

# schedule.every(1).minutes.do(func)

# while True:
#     schedule.run_pending()
#     time.sleep(1)

In [3]:
download_satellite('/Users/sarah/Documents/Blooms/bloom_geojsons/', 'jul22', 500,500, 0.59, '2025-07-22',sites_total,API_KEY)


FileNotFoundError: [Errno 2] No such file or directory: '/Users/sarah/Documents/Blooms/bloom_geojsons/'

In [28]:
nc_sites = 'may27'
dataset = DailyMonitoringDataset(root_dir=nc_sites)
may27 = DataLoader(dataset, batch_size=1, shuffle = False)
model = CNN()
MODEL_PATH = 'planet_binary.pt'
model.load_state_dict(torch.load(MODEL_PATH, 
map_location=torch.device('cpu')))

<All keys matched successfully>

21it [00:01, 10.56it/s]


In [32]:
tracking

{'date': ['05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025',
  '05-25-2025'],
 'pred': [False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False,
  False]}